In [1]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2
from prompts.medical_prompts import NumericPrompts

from rate_limiter.rate_limiter import RateLimiter

from tqdm.asyncio import tqdm as tqdm_asyncio

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [3]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_67968/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [4]:
df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI']]

In [ ]:
df = df.sample(200)
# df = df.iloc[[4581,16565]]
df

In [19]:
# tst = df.index
tst

Index([10456, 13176, 27860,  5550,  1965,  5693,  1217,  5416, 11918,  7052,
       23021,  1279, 20473,   964, 15255, 12647, 21898, 24004, 21644,  7092,
       16864,  8041, 22474, 11219, 21370, 16400, 20410, 22642, 22646,    15,
        2014, 16103,  5308, 10501, 13638, 18044,  6412, 11842, 11457,  9244,
       27466,  2201, 19473, 10868,  9870, 16037, 11764, 23617,  1439, 18418,
       16983, 23078, 13077,  5916,  6650, 26379, 15917, 27538, 12235, 16722,
       14093,  7382, 13738,  1079, 21726, 19921,  4672, 26050,  6019, 12597,
       18911,  6916,  8744,  2971, 11599,  6863, 18326,  7822,  4176, 16541,
       11727, 24261,  2511, 20792, 25828, 24799, 12668, 26712, 13674, 21260,
       26303, 25958, 27151, 16105, 22932,  2403, 22687,  9379, 21130,  8247],
      dtype='int64')

In [5]:
df = df.iloc[[10456, 13176, 27860,  5550,  1965,  5693,  1217,  5416, 11918,  7052,
       23021,  1279, 20473,   964, 15255, 12647, 21898, 24004, 21644,  7092,
       16864,  8041, 22474, 11219, 21370, 16400, 20410, 22642, 22646,    15,
        2014, 16103,  5308, 10501, 13638, 18044,  6412, 11842, 11457,  9244,
       27466,  2201, 19473, 10868,  9870, 16037, 11764, 23617,  1439, 18418,
       16983, 23078, 13077,  5916,  6650, 26379, 15917, 27538, 12235, 16722,
       14093,  7382, 13738,  1079, 21726, 19921,  4672, 26050,  6019, 12597,
       18911,  6916,  8744,  2971, 11599,  6863, 18326,  7822,  4176, 16541,
       11727, 24261,  2511, 20792, 25828, 24799, 12668, 26712, 13674, 21260,
       26303, 25958, 27151, 16105, 22932,  2403, 22687,  9379, 21130,  8247]]

In [6]:
# 여러 인덱스를 선택할 때는 리스트 형태로 전달해야 합니다
df = df.loc[[13674, 21260, 26303]]

In [7]:
df

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI
13674,2210-39,2022-11-02,"구강내과#2[도착]대기고지함)물리치료 , APS del , MMTT 필요여부 CK (PT 원하심)-> 보톡스는 나중에 놔주신다고 했던 것 같아요.증상 : 입은 좀 더 잘어지는데 하품은 잘 못해요, 음식을 먹을 때 많이 벌리긴 어려워요. 입을 벌릴 떄 통증 VAS 4-6 -> 2 정도로 줄었어요.약 : 중간에 못먹기도 했어요, 불편감 X습관 : 딱딱하고 질긴 음식 거의 안먹는데 먹고나면 턱이 아프긴 해요,찜질 : 냉찜질 2-3일정도 했어요.",NaN,NaN,NaN,NaN,NaN,"치아 교모 있음12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극측두하악관절자극요법-단순자극 10월 5일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 10월 5일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극- 분사신장치료측두하악관절자극요법-복합자극 10월 5일에 측두하악장애 분석검사 시행45678Dr.남윤진료악관절 고착 해소술 - K07.60 턱관절내장증- 악관절고착해소술12345678 12345678Dr.남윤진료동기능적 교합검사 - K07.38 치아위치의 기타 명시된 이상- 동기능적교합검사- 구강내장치 조정 및 장착- 페리슨정(에페리손염산염) - 1/1회/14일 :: 취침 직전 복용- 소론도정(프레드니솔론) - 1/1회/14일 :: 기상 직후 복용물리치료 , 장치 ck [2주후]"
21260,2203-100,2023-05-15,"구강내과#4물리치료 , 장치 ck증상: 통증, 소리 없어요. 지난주에 입 움직이다가 왼쪽턱에서 1번 약간 소리가 났는데 빠진 느낌은 아니었어요. 그 이후로는 없었어요.",NaN,장치: 매일 착용./ 불편함 없었어요. 입술테이프 20개 넘게 남았어요,습관: 딱딱하거나 질긴음식 안먹었어요./. 이랑 이가 안닿도록 노력했어요,찜질: 1주일에 2~3번 식을때까지 20~30분/ 온찜질팩,NaN,"#13-23 open bite- 장치 조정12345678 12345678Dr.박유진진료동기능적 교합검사 - K07.38 치아위치의 기타 명시된 이상- 동기능적교합검사동기능적교합검사 T-scan 이용하여 동기능적교합검사 실시함12345678 12345678Dr.박유진진료SS imp- SS Splint 인상채득12345678 12345678Dr.박유진진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극측두하악관절자극요법-단순자극 2월 23일에 측두하악장애 분석검사 시행12345678 12345678Dr.박유진진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 2월 23일에 측두하악장애 분석검사 시행12345678 12345678Dr.박유진진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극측두하악관절자극요법-복합자극 2월 23일에 측두하악장애 분석검사 시행물리치료 , 장치ck, SS del [3주후]"
26303,2206-95,2023-11-06,"물리치료 , SS re del구강내과#16증상: 입 벌릴 때 왼쪽 걸리는 느낌 아직도 약간 있어요. 소리는 안나고 턱 통증도 없어요.",NaN,장치:,습관: 딱딱하고 질긴 음식 안먹었어요.치아끼리 안닿게 했어요.,찜질:온찜질은 한번도 못했어요.,"마사지,스트레칭: 마사지랑 스트레칭 둘다 일주일에 2~3번씩 1회당 5분정도 하고있어요.","12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극측두하악관절자극요법-단순자극 6월 9일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 6월 9일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극측두하악관절자극요법-복합자극 6월 9일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료동기능적 교합검사 - K07.38 치아위치의 기타 명시된 이상- 동기능적교합검사동기능적교합검사 T-scan 이용하여 동기능적교합검사 실시함12345678 12345678Dr.남윤진료SS re-del- 구강내장치 조정 및 장착물리치료 , 장치 ck [2개월후]"


In [6]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    MODEL_NAME = "gpt-3.5-turbo"
    # MODEL_NAME = "o3-mini"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 150 #100
    SEMAPHORE_LIMIT = 8 #5
    MAX_RETRIES = 4
    LOG_FILE = "medical_classifier.log"
    # RPM_LIMIT = 4000  # 실제 한도보다 약간 낮게 설정
    # TPM_LIMIT = 3800000  # 실제 한도보다 약간 낮게 설정
    
    # o1 모델 API 제한 반영 (약간의 여유를 둠)
    RPM_LIMIT = 4800  # 5,000 RPM
    TPM_LIMIT = 3800000  # 4,000,000 TPM
    TPD_LIMIT = 38000000  # 40,000,000 TPD
    INDEX_COLUMNS = ['환자번호', '날짜']

#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)

#############################################
# 체크포인트 관리 클래스
#############################################



#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

class ImprovedMedicalTextClassifier:
    """의학 텍스트 분류기 - 개선된 오류 처리 버전"""
    
    def __init__(self, api_key: str, config=None):
        self.config = config if config is not None else Config
        
        # API 키 설정
        import os
        os.environ["OPENAI_API_KEY"] = api_key
        self.client = openai.OpenAI(api_key=api_key)
        
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        
        # Rate Limiter
        self.rate_limiter = RateLimiter(
            rpm_limit=self.config.RPM_LIMIT,
            tpm_limit=self.config.TPM_LIMIT
        )
        
        # 처리 대상 컬럼별 분류 함수
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리 - 개선된 오류 처리"""
        original_shape = df.shape
        processed_cols = 0
        
        logger.info(f"원본 DataFrame 인덱스 샘플: {df.index.tolist()[:5]}")
        
        # 원본 인덱스 보존
        original_index = df.index.copy()
        original_df = df.reset_index().copy()
        
        # 문자열 형태로 별도 컬럼에 보존 (debug)
        original_df['orig_index'] = original_index.map(str)
        logger.info(f"orig_index 샘플(5): {original_df['orig_index'].tolist()[:5]}")
        
        # 최종 결과를 담을 DataFrame 초기화
        result_df = original_df.copy()
        
        for column in self.classifiers.keys():
            if column not in df.columns:
                continue
            
            logger.info(f"Processing column: {column}")
            
            try:
                column_results = await self._process_column_no_checkpoint(result_df, column)
                
                if column_results is not None and not column_results.empty:
                    # column_results에는 orig_index가 인덱스로 설정되어 있음
                    derived_cols = [col for col in column_results.columns if col.startswith(f"{column}_")]
                    if derived_cols:
                        logger.info(f"[{column}]에서 {len(derived_cols)}개 파생 컬럼 생성: {derived_cols}")
                        
                        # 원본 result_df에 병합
                        for derived_col in derived_cols:
                            # 누락된 값은 빈 문자열로 처리
                            series_to_join = column_results[derived_col].fillna("")
                            result_df[derived_col] = result_df['orig_index'].map(series_to_join.to_dict()).fillna("")
                        
                        # 파생 컬럼 내용 검사
                        for derived_col in derived_cols:
                            cnt_nonempty = result_df[derived_col].astype(str).str.strip().ne('').sum()
                            logger.info(f"  -> {derived_col} - 유효 데이터: {cnt_nonempty}/{len(result_df)}")
            except Exception as e:
                logger.error(f"[{column}] 컬럼 처리 중 예외 발생: {str(e)}")
                # 오류가 발생해도 계속 진행
            
            processed_cols += 1
        
        # 원래 인덱스로 복원
        if 'index' in result_df.columns:
            result_df.drop('index', axis=1, inplace=True, errors='ignore')
        if 'orig_index' in result_df.columns:
            result_df.drop('orig_index', axis=1, inplace=True)
        
        result_df.index = original_index
        
        logger.info(f"[process_all_columns] 총 {processed_cols}개 컬럼 처리 완료.")
        return result_df

    async def _process_column_no_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트 없이 단일 컬럼 처리 - 개선된 오류 처리"""
        try:
            # 비어있지 않은 행 필터링
            mask = df[column].notna() & df[column].astype(str).str.strip().ne('')
            if not mask.any():
                logger.info(f"[{column}] 파싱 대상 텍스트가 없음")
                return pd.DataFrame()
            
            filtered_df = df.loc[mask].copy()
            logger.info(f"[{column}] 처리 대상 행 수: {len(filtered_df)}")
            
            # 처리할 텍스트 및 인덱스 준비
            texts_with_idx = [
                (idx, text, orig_idx)
                for idx, text, orig_idx in zip(filtered_df.index,
                                               filtered_df[column],
                                               filtered_df['orig_index'])
            ]
            
            results = await self._safe_process_batches(
                texts=[t for (_, t, _) in texts_with_idx],
                original_indices=[i for (i, _, _) in texts_with_idx],
                orig_indices=[o for (_, _, o) in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )
            
            # 결과가 없어도 최소한 인덱스 정보는 유지
            if not results:
                empty_results = [{"index": i, "orig_index": o} for i, _, o in texts_with_idx]
                results_df = pd.DataFrame(empty_results)
            else:
                results_df = pd.DataFrame(results)
            
            # 'orig_index'를 인덱스로
            if 'orig_index' in results_df.columns:
                results_df.set_index('orig_index', inplace=True)
            
            # 컬럼명 접두사 붙이기
            new_cols = []
            for col_name in results_df.columns:
                if col_name == 'index':  
                    new_cols.append(col_name)  # 'index' 유지
                else:
                    new_cols.append(f"{column}_{col_name}")
            
            results_df.columns = new_cols
            return results_df
        
        except Exception as e:
            logger.error(f"[{column}] 처리 중 오류: {str(e)}")
            # 오류가 발생해도 최소한 인덱스 정보는 반환
            if 'texts_with_idx' in locals():
                empty_results = [{"index": i, "orig_index": o} for i, _, o in texts_with_idx]
                results_df = pd.DataFrame(empty_results)
                
                if 'orig_index' in results_df.columns:
                    results_df.set_index('orig_index', inplace=True)
                    
                return results_df
            return pd.DataFrame()

    async def _safe_process_batches(self, texts: List[str],
                                    original_indices: List[int],
                                    orig_indices: List[str],
                                    classifier, column: str) -> List[Dict]:
        """API 호출 - 개선된 오류 처리 및 개별 텍스트 독립적 처리"""
        results = []
        
        for i, (txt, idx, orig_idx) in enumerate(zip(texts, original_indices, orig_indices)):
            # 디버깅: 각 텍스트 출력 (길면 앞부분만)
            shortened_text = txt[:80] + "..." if len(str(txt)) > 80 else txt
            logger.info(f"[{column}] {i+1}/{len(texts)} | index={idx}, orig_index={orig_idx}, text={shortened_text}")
            
            try:
                # 빈 텍스트 처리
                if not txt or not str(txt).strip():
                    results.append({"index": idx, "orig_index": orig_idx})
                    continue
                
                # 단일 텍스트 호출 시도
                try:
                    # 실제 API 호출 - 오류가 발생해도 계속 진행
                    single_result = await classifier([txt], self.semaphore)
                    
                    # 응답 확인 및 병합
                    if single_result and len(single_result) > 0:
                        item = single_result[0]
                        merged = {"index": idx, "orig_index": orig_idx}
                        if isinstance(item, dict):
                            merged.update(item)
                        results.append(merged)
                    else:
                        # 결과가 없으면 인덱스만 포함
                        results.append({"index": idx, "orig_index": orig_idx})
                    
                except Exception as e:
                    # API 호출 오류 발생 시 로그 기록 후 인덱스만 추가
                    logger.error(f"[{column}] 인덱스 {idx} API 호출 오류: {str(e)}")
                    results.append({"index": idx, "orig_index": orig_idx})
                
            except Exception as e:
                # 전체 처리 중 오류가 발생해도 건너뛰지 말고 인덱스 정보는 보존
                logger.error(f"[{column}] 인덱스 {idx} 텍스트 처리 오류: {str(e)}")
                results.append({"index": idx, "orig_index": orig_idx})
        
        return results

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 직접 호출 - 개선된 오류 처리"""
        try:
            async with semaphore:
                import httpx
                url = "https://api.openai.com/v1/chat/completions"
                headers = {
                    "Authorization": f"Bearer {self.config.API_KEY}",
                    "Content-Type": "application/json"
                }
                data = {
                    "model": self.config.MODEL_NAME,
                    "messages": [
                        {"role": "system", "content": "JSON 형식으로 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    "max_tokens": self.config.MAX_TOKENS,
                    "temperature": self.config.TEMPERATURE
                }
                
                # API 요청 시도
                try:
                    response = await httpx.AsyncClient().post(url, headers=headers, json=data, timeout=60.0)
                    response.raise_for_status()
                    
                    response_data = response.json()
                    content = response_data["choices"][0]["message"]["content"]
                    
                    total_tokens = response_data["usage"]["total_tokens"]
                    self.rate_limiter.record_request(total_tokens)
                except httpx.HTTPStatusError as e:
                    logger.error(f"HTTP 오류: {e.response.status_code} - {e.response.text}")
                    return [{}]  # 빈 객체 반환
                except Exception as e:
                    logger.error(f"API 호출 실패: {str(e)}")
                    return [{}]  # 빈 객체 반환
                
                # JSON 파싱
                result = self._validate_and_parse_json(content)
                return result
        except Exception as e:
            logger.error(f"API 요청 처리 중 예외 발생: {str(e)}")
            return [{}]  # 빈 객체 반환

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """JSON 파싱 - 향상된 오류 처리"""
        logger.info(f"원시 API 응답(앞200자): {content[:200]}...")
        
        # 빈 결과 기본값 - 완전한 실패 시에도 최소한 빈 목록 반환
        default_empty_result = [{}]
        
        try:
            # 직접 파싱 시도
            parsed = json.loads(content)
            if isinstance(parsed, list):
                return parsed
            elif isinstance(parsed, dict):
                # 단일 객체인 경우 리스트로 변환
                return [parsed]
        except json.JSONDecodeError:
            # JSON 파싱 실패, 계속 진행
            pass
        
        # 정규식으로 JSON 블록 추출 시도
        pattern = r'```json\s*([\s\S]*?)```|(\[[\s\S]*\])|(\{[\s\S]*\})'
        matches = re.findall(pattern, content)
        
        for match in matches:
            for m in match:
                m_strip = m.strip()
                if not m_strip:
                    continue
                
                try:
                    p = json.loads(m_strip)
                    if isinstance(p, list):
                        return p
                    elif isinstance(p, dict):
                        # 단일 객체인 경우 리스트로 변환
                        return [p]
                except json.JSONDecodeError:
                    continue
        
        # 마지막 수단: 콤마 있는 JSON 객체 찾기
        try:
            # {...} 형태의 객체 찾기
            obj_pattern = r'\{[^{}]*\}'
            obj_matches = re.findall(obj_pattern, content)
            
            if obj_matches:
                for obj_str in obj_matches:
                    try:
                        obj = json.loads(obj_str)
                        return [obj]
                    except:
                        continue
        except:
            pass
        
        logger.warning("JSON 파싱 실패(할루시네이션 가능) - 빈 객체 반환")
        return default_empty_result
    
    # 각 분류 함수도 오류 처리 강화
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
            history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
            severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
            
            combined = []
            for i in range(len(texts)):
                merged = {}
                if i < len(cc_results): merged.update(cc_results[i])
                if i < len(history_results): merged.update(history_results[i])
                if i < len(severity_results): merged.update(severity_results[i])
                combined.append(merged)
            return combined
        except Exception as e:
            logger.error(f"CC 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)  # 빈 객체 배열 반환

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            return await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        except Exception as e:
            logger.error(f"약물 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            return await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        except Exception as e:
            logger.error(f"장치 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)
    
    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            return await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        except Exception as e:
            logger.error(f"습관 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            return await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        except Exception as e:
            logger.error(f"찜질 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        try:
            return await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        except Exception as e:
            logger.error(f"마사지 분류 처리 중 오류: {str(e)}")
            return [{}] * len(texts)


#############################################
# 메디컬 데이터 처리 함수
#############################################

async def process_medical_data_improved(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """개선된 오류 처리로 의료 데이터 처리"""
    # 개선된 분류기 사용
    classifier = ImprovedMedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"처리 시작 시간: {start_time}")
    
    try:
        # 데이터프레임 복사본 생성
        df_copy = df.copy()
        
        # NaN 및 빈 문자열 처리
        for col in df_copy.columns:
            # 문자열 열은 빈 문자열로 대체, 숫자 열은 그대로 유지
            if df_copy[col].dtype == 'object':
                df_copy[col] = df_copy[col].fillna('')
        
        # 개선된 분류기로 처리
        processed_df = await classifier.process_all_columns(df_copy)
        
        # 처리 후 열별 통계 확인
        for col in processed_df.columns:
            if col in df.columns:
                # 원본 열은 건너뜀
                continue
                
            if processed_df[col].dtype == 'object':
                # 문자열 열에 대해 값 분포 확인
                non_empty = processed_df[col].astype(str).str.strip().ne('').sum()
                logger.info(f"열 '{col}': 비어있지 않은 값 {non_empty}/{len(processed_df)} ({non_empty/len(processed_df)*100:.1f}%)")
        
        end_time = datetime.now()
        duration = end_time - start_time
        logger.info(f"처리 종료 시간: {end_time}, 소요 시간: {duration}")
        
        return processed_df
    
    except Exception as e:
        end_time = datetime.now()
        duration = end_time - start_time
        logger.error(f"처리 중 오류 발생: {str(e)}, 경과 시간: {duration}")
        # 오류가 발생해도 가능한 경우 부분적으로 처리된 데이터 반환
        if 'processed_df' in locals():
            return processed_df
        return df  # 원본 데이터라도 반환


#############################################
# 메인 함수
#############################################
def main_improved():
    """개선된 오류 처리로 메인 함수 실행"""
    global logger
    logger = setup_logging()
    
    try:
        # 가정: df 라는 이미 로드된 pd.DataFrame
        original_df = df  
        
        # 비동기 루프 설정
        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            # 이벤트 루프가 없으면 새로 생성
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
        
        # 개선된 처리 함수 사용
        result_df = loop.run_until_complete(process_medical_data_improved(original_df, Config.API_KEY))
        
        # 결과 저장
        result_df.to_csv("final_result_improved.csv", index=True, encoding="utf-8-sig")
        logger.info("[main] 최종 결과 CSV 저장 완료")
        
        # 처리 통계 출력
        original_cols = set(original_df.columns)
        new_cols = set(result_df.columns) - original_cols
        
        logger.info(f"처리 전 열 개수: {len(original_cols)}")
        logger.info(f"처리 후 추가된 열 개수: {len(new_cols)}")
        logger.info(f"추가된 열 목록: {sorted(list(new_cols))}")
        
        return result_df
    
    except Exception as e:
        logger.error(f"[main] 오류 발생: {str(e)}", exc_info=True)
        # 부분적으로 처리된 결과가 있으면 반환
        if 'result_df' in locals():
            return result_df
        return None


In [7]:
import os
import pandas as pd
import numpy as np
import asyncio
import logging
from datetime import datetime
from typing import List, Dict, Optional, Tuple
import uuid

class ChunkedProcessor:
    """대규모 데이터셋을 청킹하여 처리하는 클래스"""
    
    def __init__(self, chunk_size=1000, output_dir='chunks'):
        self.chunk_size = chunk_size
        self.output_dir = output_dir
        self.processed_chunks = []
        
        # 출력 디렉토리 생성
        os.makedirs(output_dir, exist_ok=True)
        
        # 임시 처리 폴더명을 고유하게 설정
        self.process_id = uuid.uuid4().hex[:8]
        self.temp_folder = os.path.join(output_dir, f"process_{self.process_id}")
        os.makedirs(self.temp_folder, exist_ok=True)
        
        self.logger = logging.getLogger(__name__)
    
    async def process_in_chunks(self, df: pd.DataFrame, processor_func, **kwargs) -> pd.DataFrame:
        """DataFrame을 청크로 나눠 처리하고 결과를 병합하여 반환"""
        total_rows = len(df)
        total_chunks = (total_rows + self.chunk_size - 1) // self.chunk_size
        
        self.logger.info(f"총 {total_rows}행을 {total_chunks}개 청크로 나누어 처리합니다 (청크 크기: {self.chunk_size})")
        
        # 청크 생성 및 처리
        for chunk_idx in range(total_chunks):
            start_idx = chunk_idx * self.chunk_size
            end_idx = min(start_idx + self.chunk_size, total_rows)
            
            # 원본 인덱스 유지를 위해 iloc 대신 원본 인덱스로 슬라이싱
            chunk_indices = df.index[start_idx:end_idx]
            chunk_df = df.loc[chunk_indices].copy()
            
            self.logger.info(f"청크 {chunk_idx+1}/{total_chunks} 처리 중... (행 {start_idx}~{end_idx-1})")
            
            try:
                # 청크 처리
                processed_chunk = await processor_func(chunk_df, **kwargs)
                
                # 데이터 타입 정제 - parquet 저장 문제 해결
                processed_chunk = self._sanitize_dataframe(processed_chunk)
                
                # 처리된 청크를 parquet로 저장
                chunk_file = os.path.join(self.temp_folder, f"chunk_{chunk_idx:04d}.parquet")
                processed_chunk.to_parquet(chunk_file, index=True)
                self.processed_chunks.append(chunk_file)
                
                self.logger.info(f"청크 {chunk_idx+1}/{total_chunks} 처리 완료 및 저장: {chunk_file}")
                
                # 메모리 효율을 위해 청크 처리 후 명시적으로 변수 삭제
                del processed_chunk
                del chunk_df
            except Exception as e:
                self.logger.error(f"청크 {chunk_idx+1}/{total_chunks} 처리 중 오류 발생: {str(e)}")
                # 오류 발생 시에도 처리된 데이터가 있으면 저장 시도
                if 'processed_chunk' in locals() and processed_chunk is not None and not processed_chunk.empty:
                    try:
                        self.logger.info("오류 발생했지만 처리된 데이터 저장 시도...")
                        # 데이터 타입 정제 재시도
                        processed_chunk = self._sanitize_dataframe(processed_chunk)
                        
                        # CSV로 임시 저장 (Parquet 저장이 실패할 경우 대비)
                        error_csv = os.path.join(self.temp_folder, f"chunk_{chunk_idx:04d}_error.csv")
                        processed_chunk.to_csv(error_csv, index=True, encoding='utf-8-sig')
                        self.logger.info(f"오류 발생 데이터 CSV로 저장: {error_csv}")
                        
                        # Parquet 저장 재시도
                        chunk_file = os.path.join(self.temp_folder, f"chunk_{chunk_idx:04d}.parquet")
                        processed_chunk.to_parquet(chunk_file, index=True)
                        self.processed_chunks.append(chunk_file)
                        self.logger.info(f"오류 청크 파일 저장 성공: {chunk_file}")
                    except Exception as save_err:
                        self.logger.error(f"오류 데이터 저장 실패: {str(save_err)}")
                        # CSV는 저장되었지만 parquet 저장에 실패한 경우, CSV를 나중에 처리
                        if os.path.exists(error_csv):
                            self.logger.info(f"CSV 파일만 저장됨: {error_csv}")
        
        # CSV로 저장된 오류 청크 처리
        error_csvs = [f for f in os.listdir(self.temp_folder) if f.endswith('_error.csv')]
        for error_csv in error_csvs:
            try:
                csv_path = os.path.join(self.temp_folder, error_csv)
                error_df = pd.read_csv(csv_path, index_col=0)
                chunk_idx = int(error_csv.split('_')[1].split('.')[0])
                
                # 데이터 타입 정제
                error_df = self._sanitize_dataframe(error_df)
                
                # Parquet 저장 재시도
                chunk_file = os.path.join(self.temp_folder, f"chunk_{chunk_idx:04d}.parquet")
                error_df.to_parquet(chunk_file, index=True)
                
                if chunk_file not in self.processed_chunks:
                    self.processed_chunks.append(chunk_file)
                    self.logger.info(f"오류 CSV에서 복구된 청크 저장: {chunk_file}")
            except Exception as e:
                self.logger.error(f"오류 CSV 복구 실패: {str(e)}")
        
        # 모든 처리된 청크 병합
        return self._merge_processed_chunks()
    
    def _sanitize_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Parquet 저장을 위한 데이터프레임 정제"""
        if df is None or df.empty:
            return pd.DataFrame()
        
        # 각 열의 데이터 타입 체크 및 변환
        for col in df.columns:
            # 원래의 인덱스 및 특수 열은 그대로 유지
            if col == 'index' or col == 'orig_index':
                continue
                
            # 숫자형 데이터 처리 (예: 'severity', 'duration' 등)
            if any(substr in col for substr in ['_severity', '_duration', '_term', '_status']):
                # 숫자로 변환 가능한 값만 숫자로, 나머지는 NaN으로
                df[col] = pd.to_numeric(df[col], errors='coerce')
                # NaN 값은 None으로 대체
                df[col] = df[col].replace({np.nan: None})
            else:
                # 문자열 데이터는 모두 문자열로 통일, None은 빈 문자열로
                df[col] = df[col].astype(str).replace({'nan': '', 'None': ''})
                
        return df
    
    def _merge_processed_chunks(self) -> pd.DataFrame:
        """처리된 모든 청크를 하나의 DataFrame으로 병합"""
        if not self.processed_chunks:
            self.logger.warning("병합할 처리된 청크가 없습니다.")
            return pd.DataFrame()
        
        self.logger.info(f"총 {len(self.processed_chunks)}개의 청크를 병합합니다...")
        
        result_frames = []
        
        # 각 청크 로드 및 병합 준비
        for chunk_file in self.processed_chunks:
            try:
                chunk_df = pd.read_parquet(chunk_file)
                # 타입 정제 (병합 시 충돌 방지)
                chunk_df = self._sanitize_dataframe(chunk_df)
                result_frames.append(chunk_df)
            except Exception as e:
                self.logger.error(f"청크 파일 {chunk_file} 로드 중 오류 발생: {str(e)}")
        
        if not result_frames:
            self.logger.warning("병합 가능한 청크가 없습니다.")
            return pd.DataFrame()
            
        # 모든 프레임 병합
        try:
            result_df = pd.concat(result_frames, axis=0)
            # 인덱스 정렬 (원본 순서 유지를 위해)
            result_df = result_df.sort_index()
            
            self.logger.info(f"병합 완료: {len(result_df)}행")
            return result_df
        except Exception as e:
            self.logger.error(f"청크 병합 중 오류 발생: {str(e)}")
            # 병합 실패 시 첫 번째 사용 가능한 청크라도 반환
            if result_frames:
                self.logger.warning("병합 실패. 첫 번째 청크만 반환합니다.")
                return result_frames[0]
            return pd.DataFrame()
    
    def cleanup(self):
        """임시 파일 정리"""
        try:
            import shutil
            if os.path.exists(self.temp_folder):
                shutil.rmtree(self.temp_folder)
                self.logger.info(f"임시 청크 폴더 삭제 완료: {self.temp_folder}")
        except Exception as e:
            self.logger.warning(f"임시 파일 정리 중 오류 발생: {str(e)}")

# 기존 코드에 통합하기 위한 수정된 처리 함수
async def process_medical_data_chunked(df: pd.DataFrame, api_key: str, chunk_size=1000) -> pd.DataFrame:
    """Chunked processing으로 대규모 의료 데이터 처리"""
    setup_logging()  # 로깅 설정
    logger = logging.getLogger(__name__)
    
    start_time = datetime.now()
    logger.info(f"청크 처리 시작 시간: {start_time}")
    
    # 청크 프로세서 초기화
    chunk_processor = ChunkedProcessor(chunk_size=chunk_size)
    
    try:
        # 청크 단위로 처리
        result_df = await chunk_processor.process_in_chunks(
            df, 
            process_medical_data_improved,  # 기존 처리 함수 사용
            api_key=api_key
        )
        
        # 결과가 비어있으면 원본 데이터 복사본 반환
        if result_df.empty and not df.empty:
            logger.warning("처리 결과가 비어있습니다. 원본 데이터의 복사본을 반환합니다.")
            result_df = df.copy()
        
        # 결과 저장 (최종 통합 결과)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # 타입 정제 후 저장 시도
        try:
            result_df = chunk_processor._sanitize_dataframe(result_df)
            
            # Parquet 저장
            result_parquet = f"./final_result/final_result_{timestamp}.parquet"
            result_df.to_parquet(result_parquet, index=True)
            logger.info(f"Parquet 결과 저장 완료: {result_parquet}")
        except Exception as e:
            logger.error(f"Parquet 저장 중 오류 발생: {str(e)}")
        
        # CSV 형식으로도 저장 (기존 코드와의 호환성)
        try:
            result_csv = f"./final_result/final_result_{timestamp}.csv"
            result_df.to_csv(result_csv, index=True, encoding="utf-8-sig")
            logger.info(f"CSV 결과 저장 완료: {result_csv}")
        except Exception as e:
            logger.error(f"CSV 저장 중 오류 발생: {str(e)}")
        
        # 청크 임시 파일 정리
        chunk_processor.cleanup()
        
        end_time = datetime.now()
        duration = end_time - start_time
        logger.info(f"총 처리 시간: {duration}")
        
        return result_df
    
    except Exception as e:
        logger.error(f"청크 처리 중 오류 발생: {str(e)}", exc_info=True)
        # 가능한 경우 부분적으로 처리된 데이터 반환
        try:
            # 현재까지 처리된 청크 병합 시도
            partial_result = chunk_processor._merge_processed_chunks()
            if not partial_result.empty:
                logger.info(f"부분 결과 반환: {len(partial_result)}행")
                return partial_result
            else:
                logger.warning("병합 가능한 결과가 없습니다. 원본 데이터 반환")
                return df
        except:
            logger.error("부분 결과 병합 실패, 원본 데이터 반환")
            return df

# 수정된 메인 함수
def main_chunked_improved(chunk_size=1000):
    """청킹을 지원하는 메인 함수"""
    logger = setup_logging()
    
    try:
        # 가정: df 라는 이미 로드된 pd.DataFrame
        original_df = df  
        
        # 비동기 루프 설정
        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            # 이벤트 루프가 없으면 새로 생성
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
        
        # 청킹 처리 함수 사용
        result_df = loop.run_until_complete(
            process_medical_data_chunked(original_df, Config.API_KEY, chunk_size=chunk_size)
        )
        
        # 처리 통계 출력
        original_cols = set(original_df.columns)
        new_cols = set(result_df.columns) - original_cols
        
        logger.info(f"처리 전 열 개수: {len(original_cols)}")
        logger.info(f"처리 후 추가된 열 개수: {len(new_cols)}")
        logger.info(f"추가된 열 목록: {sorted(list(new_cols))}")
        
        return result_df
    
    except Exception as e:
        logger.error(f"[main] 오류 발생: {str(e)}", exc_info=True)
        # 부분적으로 처리된 결과가 있으면 반환
        if 'result_df' in locals():
            return result_df
        return df

if __name__ == "__main__":
    result = main_chunked_improved(chunk_size=100)

2025-04-07 21:11:36,705 - __main__ - INFO - 청크 처리 시작 시간: 2025-04-07 21:11:36.705318
2025-04-07 21:11:36,737 - __main__ - INFO - 총 200행을 2개 청크로 나누어 처리합니다 (청크 크기: 100)
2025-04-07 21:11:36,794 - __main__ - INFO - 청크 1/2 처리 중... (행 0~99)
2025-04-07 21:11:36,819 - __main__ - INFO - 처리 시작 시간: 2025-04-07 21:11:36.819348
2025-04-07 21:11:36,824 - __main__ - INFO - 원본 DataFrame 인덱스 샘플: [28060, 27949, 27424, 27461, 19789]
2025-04-07 21:11:36,827 - __main__ - INFO - orig_index 샘플(5): ['28060', '27949', '27424', '27461', '19789']
2025-04-07 21:11:36,828 - __main__ - INFO - Processing column: CC
2025-04-07 21:11:36,885 - __main__ - INFO - [CC] 처리 대상 행 수: 100
2025-04-07 21:11:36,886 - __main__ - INFO - [CC] 1/100 | index=0, orig_index=28060, text=말하거나 식사할때 턱에서 소리가나요 전기가 오듯이 찌릿찌릿해요 식사를 거의못해요2년 전부터말할 때, 식사할 때 오른쪽 입천장 안쪽 이 욱씬거려요...
2025-04-07 21:11:39,648 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-04-07 21:11:39,654 - __main__ - INFO - 원시 API 응

In [15]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/final_result_20250406_205356.parquet")

""


In [8]:
## 에러 확인 
def diagnose_errors(log_file=Config.LOG_FILE, output_file="error_diagnosis.csv"):
    """로그 파일을 분석하여 오류를 진단하고 CSV로 출력"""
    import re
    import pandas as pd
    from collections import defaultdict
    
    # 오류 패턴 정의
    error_patterns = [
        (r"\[([^\]]+)\] 인덱스 (\d+) 텍스트 처리 오류: (.+)", "텍스트 처리 오류"),
        (r"\[([^\]]+)\] 인덱스 (\d+) API 호출 오류: (.+)", "API 호출 오류"),
        (r"JSON 파싱 실패\(할루시네이션 가능\)", "JSON 파싱 실패"),
        (r"HTTP 오류: (\d+) - (.+)", "HTTP 오류"),
        (r"API 호출 실패: (.+)", "API 호출 실패"),
        (r"API 요청 처리 중 예외 발생: (.+)", "API 요청 처리 오류"),
        (r"\[([^\]]+)\] 처리 중 오류: (.+)", "컬럼 처리 오류"),
    ]
    
    # 오류 정보를 저장할 딕셔너리
    errors = defaultdict(list)
    current_context = ""
    
    # 로그 파일 읽기
    try:
        with open(log_file, 'r', encoding='utf-8') as f:
            log_lines = f.readlines()
    except Exception as e:
        print(f"로그 파일 읽기 오류: {str(e)}")
        return
    
    # 로그 분석
    for line in log_lines:
        # 현재 컨텍스트 업데이트 (인덱스, 텍스트 정보)
        context_match = re.search(r"\[([^\]]+)\] \d+/\d+ \| index=(\d+), orig_index=([^,]+), text=(.+)", line)
        if context_match:
            column, index, orig_index, text = context_match.groups()
            current_context = f"{column}|{index}|{orig_index}|{text}"
            continue
        
        # 오류 패턴 검색
        for pattern, error_type in error_patterns:
            match = re.search(pattern, line)
            if match:
                timestamp = re.search(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})", line)
                timestamp_str = timestamp.group(1) if timestamp else ""
                
                if error_type == "텍스트 처리 오류" or error_type == "API 호출 오류" or error_type == "컬럼 처리 오류":
                    column = match.group(1)
                    if "인덱스" in pattern:
                        index = match.group(2)
                        error_msg = match.group(3)
                    else:
                        index = "N/A"
                        error_msg = match.group(2)
                    
                    errors[error_type].append({
                        "timestamp": timestamp_str,
                        "column": column,
                        "index": index,
                        "error": error_msg,
                        "context": current_context
                    })
                else:
                    # 다른 유형의 오류
                    if "HTTP 오류" in error_type:
                        status_code = match.group(1)
                        error_msg = match.group(2)
                    else:
                        if len(match.groups()) > 0:
                            error_msg = match.group(1)
                        else:
                            error_msg = "상세 정보 없음"
                    
                    errors[error_type].append({
                        "timestamp": timestamp_str,
                        "error": error_msg,
                        "context": current_context
                    })
                break
    
    # 오류 요약
    print("=== 오류 요약 ===")
    for error_type, error_list in errors.items():
        print(f"{error_type}: {len(error_list)}건")
    
    # 각 오류 유형별로 CSV 저장
    for error_type, error_list in errors.items():
        if error_list:
            df = pd.DataFrame(error_list)
            type_safe = error_type.replace("/", "_").replace(" ", "_")
            df.to_csv(f"{type_safe}_{output_file}", index=False, encoding="utf-8-sig")
            print(f"{error_type} 오류 {len(error_list)}건을 {type_safe}_{output_file}로 저장했습니다.")
    
    return errors

diagnose_errors()

=== 오류 요약 ===
텍스트 처리 오류: 77건
JSON 파싱 실패: 152건
HTTP 오류: 14건
텍스트 처리 오류 오류 77건을 텍스트_처리_오류_error_diagnosis.csv로 저장했습니다.
JSON 파싱 실패 오류 152건을 JSON_파싱_실패_error_diagnosis.csv로 저장했습니다.
HTTP 오류 오류 14건을 HTTP_오류_error_diagnosis.csv로 저장했습니다.


defaultdict(list,
            {'텍스트 처리 오류': [{'timestamp': '2025-03-30 20:41:13,810',
               'column': 'CC',
               'index': '0',
               'error': 'RetryError[<Future at 0x13dc44230 state=finished raised HTTPStatusError>]',
               'context': 'CC|0|4581|물리치료 , 장치 ck구강내과#7증상: 저번이랑 비슷하게 주 1회정도 뻐근함이 있어요, 지금은 입가쪽으로 위치가 변했어요.       보통 인식...'},
              {'timestamp': '2025-03-30 20:41:19,668',
               'column': 'CC',
               'index': '1',
               'error': 'RetryError[<Future at 0x13dc44e60 state=finished raised HTTPStatusError>]',
               'context': 'CC|1|16565|[도착]물리치료 , SS del증상이 많이 좋아졌어요구강내과#2 림증상: 통증 없고 , 턱움직여도 이상 없어요        입 안다물어진적은 없...'},
              {'timestamp': '2025-03-30 20:41:25,395',
               'column': '장치',
               'index': '0',
               'error': 'RetryError[<Future at 0x12f15f5f0 state=finished raised HTTPStatusError>]',
               'context': '장치|0|4581|장치: 주 1회 착용, 밴드O, 장치 불편감 : 여전히 아랫니가

In [58]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
4581,0,입가쪽,뻐근함,턱 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","뻐근함이 있어요, 위치가 변했어요, 인식을 못할때는 괜찮은데 한번 인식을 하면 쭉 그래요",물리치료,None,7,None


In [17]:
original_index = df.index.copy()
original_index
original_df = df.reset_index()
original_df['orig_index'] = original_index.map(str)
original_index = df.index.copy()
original_df = df.reset_index().copy()  # 인덱스를 컬럼으로 변환하여 복사

# 원본 인덱스를 문자열로 보존
original_df['orig_index'] = original_index.map(str)
original_df
result_df = original_df.copy()
result_df

classifier = MedicalTextClassifier(api_key)
classifier.process_all_columns(df)

In [47]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
0,0,턱,통증,통증-전혀 없었어요,,스트레칭-10번씩 거의 매일 했어요,물리치료,"턱관절장애 관련 과거 병력, 치료 이력",딱딱하고 질긴거 그냥 먹었어요. 치아끼리 안물려고 노력했어요.,물리치료,NaN,None,None
1,1,"왼쪽 어금니, 위 앞니, 아랫니","딱딱한 소리, 귓속 통증, 어금니 통증, 앞니 금이, 아랫니 깎임","딱 소리, 귓속 통증, 어금니 통증","앞니 금이, 아랫니 깎임",운동 시 더 아픔,보톡스,"왼쪽에서 딱 소리나고(최근), 양쪽 다 음식먹을때, 입벌릴때 귓속이 아파요. 오른쪽...",질기고 딱딱한거 많이 피하지는 않았고 치아 물지 않으려고 했어요.,보톡스,2.0,None,None


### merge

In [23]:
sens = pd.read_parquet('processed_medical_data_20250310_023712.parquet')
nums = pd.read_parquet('../../data/centum_data_numeric_cleaned.parquet')

In [24]:
len(nums), len(sens)


(28108, 28108)

In [25]:
nums.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel', 'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_bef

In [26]:
sens = sens[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method']]

In [27]:
fin_df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')
cols = [
       '환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days'
]
fin_df = fin_df[cols]
fin_df.head()

fin_df.to_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')
fin_df.to_csv('../../data/final_without_pi_centum_data_with_medical_data.csv', index=False, encoding='utf-8-sig')